# Задача 02. Категории товаров: выручка, прибыль и маржинальность

**Постановка задачи:** нужно понять, какие категории дают больше денег и какие категории наиболее маржинальны. Для доставленных заказов посчитать:

- количество проданных единиц;
- выручку;
- валовую прибыль;
- маржинальность в процентах;
- среднюю скидку.

Отсортировать категории по выручке по убыванию.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path('../data/marketplace.sqlite')
conn = sqlite3.connect(DB_PATH)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print(f'База подключена: {DB_PATH.resolve()}')

In [ ]:
query = r"""
SELECT
    p.category,
    SUM(oi.quantity) AS units_sold,
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_pct / 100.0)), 2) AS revenue,
    ROUND(SUM(oi.quantity * (oi.unit_price * (1 - oi.discount_pct / 100.0) - p.cost)), 2) AS gross_profit,
    ROUND(
        100.0 * SUM(oi.quantity * (oi.unit_price * (1 - oi.discount_pct / 100.0) - p.cost))
        / NULLIF(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_pct / 100.0)), 0),
        2
    ) AS margin_pct,
    ROUND(AVG(oi.discount_pct), 2) AS avg_discount_pct
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
WHERE o.status = 'delivered'
GROUP BY p.category
ORDER BY revenue DESC;
"""

df = pd.read_sql_query(query, conn)
df

**Комментарий стажёра:** в этой задаче важно разделять абсолютную прибыль и маржинальность. Категория может давать большую выручку, но иметь среднюю или низкую маржу.